# ogbn-arxiv node classification: run the experiment grid on Colab

Runs `src/node_classification/ogb_run.py` for a list of (model, loss) configurations,
ten seeds each, and copies the outputs to Google Drive every two minutes and after
every run. The same script runs in a terminal at home (`scripts/train_arxiv_all.sh`),
so the grid can be split between the two: put some run strings in `RUNS` here and
start the others locally. Outputs are tagged `<model>_<loss>`, so the two sides never
overwrite each other and the folders merge by copying.

**Before running on Colab**

1. Commit and push: the notebook clones `origin/main`, so everyone who opens it
   runs the same code. Cell 3 stops if the clone is older than this notebook.
2. Select Kernel → Colab → New → GPU (T4 is enough for GraphSAGE; GATv2 at
   full batch is memory-heavy, see the `RUNS` cell).
3. Optional: set `USE_DRIVE = True` to keep results across runtime resets.

Every run starts clean: `ogb_run.py` deletes that tag's old files before seed 0,
locally and (in `run_list`) on Drive, and rewrites results, history and summary
after every seed. Nothing from an earlier attempt can show up as this run's
result.

**After a runtime reset**: run cells 1 to 4 again, then the run cell as it is.
The run cells pass `resume=True`, so a run whose seeds were backed up to Drive
continues from the last finished seed and prints which seeds it kept. A run whose
flags differ from the backed-up one is reported and left untouched; with nothing
to continue the run simply starts fresh.

Locally the notebook also works: skip the clone/install cells, the project root
is found from the working directory.

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
USE_DRIVE = True          # Colab only: mirror outputs/ to Drive after every run
REPO_URL = "https://github.com/Wb-az/pyg-graph-networks.git"
BRANCH = "main"

if IN_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE_DIR = Path("/content/drive/MyDrive/GraphNetworks")
        DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    # The code always comes from origin/main. Push before running, so anyone
    # opening this notebook runs the same code. Training uses the runtime's
    # local disk; Drive is slow for the many small checkpoint writes.
    repo = Path("/content/pyg-graph-networks")
    if not repo.exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(repo)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only"], check=True)
    os.chdir(repo)
    print(subprocess.run(["git", "log", "-1", "--format=code at %h  %s  (%cd)"],
                         capture_output=True, text=True).stdout.strip())
else:
    USE_DRIVE = False
    # Local: find the project root (pyproject.toml) from the notebook location.
    here = Path.cwd()
    root = next(p for p in [here, *here.parents] if (p / "pyproject.toml").exists())
    os.chdir(root)

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("project root:", ROOT)


In [2]:
# Colab has torch + CUDA preinstalled; add the graph stack on top.
# pyg-lib is not needed: every run below trains full batch (--no-dataloader).
if IN_COLAB:
    %pip install -q torch_geometric==2.8.0.post1 ogb pytorch-focalloss

In [ ]:
# The clone must carry the ogb_run.py this notebook was written against.
OGB_RUN = ROOT / "src" / "node_classification" / "ogb_run.py"
if not OGB_RUN.exists() or "def resume_problem" not in OGB_RUN.read_text():
    raise RuntimeError(f"{OGB_RUN} is missing or older than this notebook: "
                       "commit, push, then rerun from the top")
print(OGB_RUN, "ok")


In [ ]:
import torch
from src.node_classification.ogb_run import run_from_flags
from src.node_classification.best_model import load_summaries

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info()
    print(f"GPU: {torch.cuda.get_device_name(0)}  free {free / 1e9:.1f} / {total / 1e9:.1f} GB")
elif IN_COLAB:
    print("No GPU: Select Kernel -> Colab -> New -> GPU, then rerun from the top")

METRICS_DIR = ROOT / "outputs" / "metrics" / "ogbn-arxiv"
CKPT_DIR = ROOT / "outputs" / "checkpoints" / "ogbn-arxiv"
METRICS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

def sync(src: Path, dst: Path):
    """Copy a directory tree; files in src overwrite dst (dirs_exist_ok merges)."""
    if src.exists():
        shutil.copytree(src, dst, dirs_exist_ok=True)

if USE_DRIVE:
    DRIVE_METRICS = DRIVE_DIR / "metrics" / "ogbn-arxiv"
    DRIVE_CKPT = DRIVE_DIR / "checkpoints" / "ogbn-arxiv"
    DRIVE_METRICS.mkdir(parents=True, exist_ok=True)
    DRIVE_CKPT.mkdir(parents=True, exist_ok=True)
    # Bring back finished runs (skipped by run_list) and the seeds of an
    # interrupted one (continued by run_list(..., resume=True)).
    sync(DRIVE_METRICS, METRICS_DIR)
    sync(DRIVE_CKPT, CKPT_DIR)
    print("restored from Drive:", sorted(p.name for p in METRICS_DIR.glob("*_config.json")))

## Runs

## Runs

`BASE` follows the OGB reference configuration described in `docs/project_report.md`, section 6.1: 3 GNN layers, 256 hidden channels, dropout 0.5, learning rate 0.01, Adam optimisation with weight decay `5e-4`, a maximum of 500 epochs, validation-loss early stopping with patience 50, full-batch training, and ten random seeds. The learning rate is held constant by disabling the scheduler.

The core factorial comparison evaluates **three architectures** — GraphSAGE, GraphSAGE with Batch Normalisation (SAGEBN), and GATv2 — under **two loss functions**:

* standard cross-entropy (CE);
* tempered weighted cross-entropy (`weight_power=0.5`).

This produces a consistent architecture × loss comparison while holding the remaining training configuration fixed.

An additional GraphSAGE experiment uses untempered weighted cross-entropy. This acts as a sensitivity test for the strength of class rebalancing and allows comparison between no weighting, tempered weighting, and full class weighting. It is treated as an additional experiment rather than part of the core 3 × 2 factorial design.

Focal loss was excluded from the main experiment following the pilot described in `docs/project_report.md`, section 6.3. The final experiment therefore focuses on the more interpretable comparison between standard CE and weighted CE while avoiding an unnecessary additional computational dimension.

### GATv2 configuration

GATv2 requires a smaller per-head hidden dimension because `GATv2Conv` materialises attention-related tensors across edges, heads, and hidden features. With 8 attention heads and 32 hidden channels per head, the concatenated representation has a total width of 256 while remaining feasible on higher-memory GPUs.

The primary GATv2 configuration is therefore:

* 8 attention heads;
* 32 hidden channels per head;
* 256 total concatenated representation width.

This configuration is intended for an L4 or A100 GPU. A reduced 16-hidden-channel configuration can be used on a T4 where memory constraints prevent the 32-channel model from running.

### Execution and reproducibility

Each principal configuration is evaluated over ten seeds. Completed runs are identified using the corresponding `<tag>_config.json` file and are skipped only when its status is recorded as `done`. The presence of a summary CSV alone is not considered evidence that a run completed because `ogb_run.py` updates summary outputs after individual seeds.

The run cells call `run_list(..., resume=True)`: an interrupted experiment continues from its last completed seed when the backed-up flags match, is reported and left untouched when they differ, and starts fresh when nothing was backed up. `resume=False` removes the existing files for the run tag and starts over.

Experiment outputs are periodically copied to Google Drive while the run loop is active and again after each completed configuration. This reduces the amount of work lost if a Colab runtime terminates during training.

### Experiment grid

| Architecture          | Cross-entropy | Tempered weighted CE |
| --------------------- | :-----------: | :------------------: |
| GraphSAGE             |       ✓       |           ✓          |
| GraphSAGE + BatchNorm |       ✓       |           ✓          |
| GATv2                 |       ✓       |           ✓          |

Additional sensitivity experiment:

| Architecture | Loss                              |
| ------------ | --------------------------------- |
| GraphSAGE    | Untempered weighted cross-entropy |


In [6]:
import json

BASE = ("--dataset ogbn-arxiv --num_layers 3 --hidden_channels 256 --dropout 0.5 --lr 0.01 "
        "--epochs 500 --early_stop 501 --no-scheduler --no-dataloader --log_steps 50")
WCE = "--loss weighted_ce --weight_power 0.5"   # tempered weights, tag suffix _p0.5

GAT_WIDTH = 32   # 8 x 32 = 256 needs an L4 or A100; use 16 on a T4
GATV2 = [
    f"{BASE} --model GATV2  --heads 8 --hidden_channels {GAT_WIDTH} --loss cross_entropy",
    f"{BASE} --model GATV2  --heads 8 --hidden_channels {GAT_WIDTH} {WCE}",
]
SAGEBN = [
    f"{BASE} --model SAGEBN --loss cross_entropy",
    f"{BASE} --model SAGEBN {WCE}",
]
SAGE = [   # A1 rerun, the tempered weighted run, then balanced weights (over-correction row)
    f"{BASE} --model SAGE --loss cross_entropy",
    f"{BASE} --model SAGE {WCE}",
    f"{BASE} --model SAGE --loss weighted_ce",
]

def run_tag(flags: str) -> str:
    """The prefix ogb_run.py puts on every output file (its output_tag)."""
    from src.node_classification.ogb_run import build_parser, output_tag
    return output_tag(build_parser().parse_args(flags.split()))

def run_done(tag: str) -> bool:
    """Finished runs only: ogb_run.py sets status done after the last seed."""
    path = METRICS_DIR / f"{tag}_config.json"
    return path.exists() and json.loads(path.read_text()).get("status") == "done"

for flags in GATV2 + SAGEBN + SAGE:
    tag = run_tag(flags)
    print(("done   " if run_done(tag) else "pending"), tag)


In [ ]:
import time, traceback, threading
from src.node_classification.ogb_run import (build_parser, clear_tag_outputs,
                                             resume_problem, validate_args)

def backup():
    if USE_DRIVE:
        sync(METRICS_DIR, DRIVE_METRICS)
        sync(CKPT_DIR, DRIVE_CKPT)

def cannot_resume(flags: str, tag: str):
    """None if the restored <tag> files continue under these flags, else the reason."""
    parser = build_parser()
    args = validate_args(parser, parser.parse_args(flags.split()))
    return resume_problem(METRICS_DIR / f"{tag}_config.json",
                          METRICS_DIR / f"{tag}_results.csv", args)

def run_list(runs, resume=True, backup_every_s=120):
    """Train each configuration in turn. Finished runs are skipped.
    resume=True continues interrupted runs from their last finished seed
    (restored from Drive in the setup cell); a run whose flags differ is
    reported and left untouched. resume=False starts every run clean.
    Outputs go to Drive every backup_every_s seconds and after every run."""
    stop = threading.Event()

    def keep_backing_up():
        while not stop.wait(backup_every_s):
            try:
                backup()
            except OSError as err:          # Drive hiccup: try again next round
                print("backup failed:", err)

    threading.Thread(target=keep_backing_up, daemon=True).start()
    failed = {}
    try:
        for flags in runs:
            if resume and "--resume" not in flags:
                flags += " --resume"
            tag = run_tag(flags)
            if run_done(tag):
                print(f"skip {tag}: finished (config status done)")
                continue
            if "--resume" in flags:
                if (METRICS_DIR / f"{tag}_config.json").exists():
                    problem = cannot_resume(flags, tag)
                    if problem:
                        failed[tag] = problem
                        print(f"CANNOT RESUME {tag}: {problem}\n  nothing deleted; "
                              "fix the flags, or use resume=False to start over")
                        continue
            elif USE_DRIVE:
                # ogb_run.py deletes the tag's old files locally; mirror that on
                # Drive so a backup never mixes old and new files of one tag.
                for path in clear_tag_outputs(DRIVE_METRICS, DRIVE_CKPT, "ogbn-arxiv", tag):
                    print("removed from Drive:", path.name)
            print("=" * 100)
            print(f"start {tag}")
            t0 = time.time()
            try:
                summary, _ = run_from_flags(flags)
                print(summary)
            except (RuntimeError, MemoryError) as err:      # CUDA OOM is a RuntimeError
                failed[tag] = repr(err)
                traceback.print_exc()
                print(f"FAILED {tag}; continuing with the next run")
            finally:
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                print(f"{tag}: {(time.time() - t0) / 60:.1f} min")
                backup()
                if USE_DRIVE:
                    print("backed up to", DRIVE_DIR)
    finally:
        stop.set()
        backup()
    if failed:
        print("\nRuns that did not finish:")
        for tag, err in failed.items():
            print(f"  {tag}: {err[:300]}")
    return failed


In [ ]:
# Run 1: GraphSAGE, cross entropy then weighted CE. Continues an interrupted run; resume=False starts over.
run_list(SAGE, resume=True)


In [ ]:
# Run 2: GraphSAGE-BN, cross entropy then weighted CE.
run_list(SAGEBN, resume=True)


In [ ]:
# Run 3: GATV2, cross entropy then weighted CE. Watch the first seed for an out-of-memory error.
run_list(GATV2, resume=True)


In [ ]:
# Every finished run in this metrics folder (terminal runs too, once copied in).
table = load_summaries(METRICS_DIR)
table[["acc", "f1", "balanced_accuracy", "ece", "loss"]].round(4)

## Bringing the results home

Copy `GraphNetworks/metrics/ogbn-arxiv/` and `GraphNetworks/checkpoints/ogbn-arxiv/`
from Drive into the local `outputs/metrics/ogbn-arxiv/` and `outputs/checkpoints/ogbn-arxiv/`.
The files are tagged `<model>_<loss>`, so they sit next to the terminal runs without
renaming. Then, locally:

```
uv run python src/node_classification/compare_best.py --dataset ogbn-arxiv
uv run python src/node_classification/visualise_best.py --dataset ogbn-arxiv --metric f1
```

and fill in section 6 of `docs/project_report.md`.